In [ ]:
!pip install yfinance plotly pandas openpyxl

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
import sqlite3
import plotly.io as pio
from google.colab import files

pio.renderers.default = 'colab'

In [ ]:
uploaded = files.upload()

In [ ]:
gpr = pd.read_excel(list(uploaded.keys())[0])
gpr = gpr[['month', 'GPR']]
gpr['month'] = pd.to_datetime(gpr['month'])
gpr['month'] = gpr['month'].dt.to_period('M').dt.to_timestamp()
gpr = gpr.rename(columns={'month': 'date'})
gpr.set_index('date', inplace=True)
print(gpr.tail())

In [ ]:
cpi = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL",
                  parse_dates=['observation_date'], index_col='observation_date')
cpi.index.name = 'date'
cpi.columns = ['CPI']
cpi['inflation_yoy'] = cpi['CPI'].pct_change(12) * 100
cpi = cpi[['inflation_yoy']].dropna()
print(cpi.tail())

In [ ]:
sp500 = yf.download('^GSPC', start='1985-01-01', end='2024-12-31', auto_adjust=True)['Close']
sp500 = sp500.resample('MS').last()
sp500.columns = ['SP500']
sp500['sp500_return'] = sp500['SP500'].pct_change() * 100
print(sp500.tail())

In [ ]:
oil = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO",
                  parse_dates=['observation_date'], index_col='observation_date')
oil.index.name = 'date'
oil.columns = ['WTI']
oil['WTI'] = pd.to_numeric(oil['WTI'], errors='coerce')
oil = oil.resample('MS').mean()
oil['oil_return'] = oil['WTI'].pct_change() * 100
print(oil.tail())

In [ ]:
df = gpr.join([cpi, sp500[['sp500_return']], oil[['oil_return', 'WTI']]], how='inner')
df.dropna(inplace=True)
df.columns = ['Geopolitical Risk', 'Inflation (%)', 'SP500 Return (%)',
              'Oil Return (%)', 'Oil Price ($/barrel)']
print(df.shape)
print(df.tail())

In [ ]:
df.to_csv('master_dataset.csv')

conn = sqlite3.connect('geopolitical_risk.db')
df.to_sql('macro_data', conn, if_exists='replace', index=True)

result = conn.execute("SELECT COUNT(*) as total_rows FROM macro_data").fetchone()
print(f"Rows in database: {result[0]}")

sample = pd.read_sql("SELECT * FROM macro_data ORDER BY \"index\" DESC LIMIT 5", conn)
print(sample)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sqlite3

conn = sqlite3.connect('geopolitical_risk.db')
df = pd.read_sql("SELECT * FROM macro_data", conn)
df = df.rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
print(df.tail())

In [ ]:

crises = {
    'Gulf War':     '1990-08-01',
    '9/11':         '2001-09-01',
    'Iraq War':     '2003-03-01',
    'Financial Crisis': '2008-09-01',
    'Crimea':       '2014-03-01',
    'COVID-19':     '2020-03-01',
    'Ukraine War':  '2022-02-01',
}

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Geopolitical Risk'],
    fill='tozeroy', fillcolor='rgba(220,50,50,0.15)',
    line=dict(color='crimson', width=1),
    name='Geopolitical Risk'
))

for label, date in crises.items():
    fig.add_vline(x=date, line_dash='dash', line_color='black', line_width=0.8, opacity=0.5)
    fig.add_annotation(x=date, y=df['Geopolitical Risk'].max()*0.95,
                       text=label, showarrow=False,
                       textangle=-90, font=dict(size=9), xanchor='right')

fig.update_layout(
    title=dict(text='How Worried Was the World? Geopolitical Risk Index, 1985–2024',
               font=dict(size=16)),
    xaxis_title='', yaxis_title='Geopolitical Risk Index',
    hovermode='x unified',
    template='plotly_white',
    height=450
)

from IPython.display import display, HTML
display(HTML(fig.to_html()))
fig.write_html('chart1_geopolitical_risk.html')
#print("Saved!")

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Inflation (%)'],
    fill='tozeroy', fillcolor='rgba(255,140,0,0.15)',
    line=dict(color='darkorange', width=1.5),
    name='Inflation YoY %'
))

fig.add_hline(y=2, line_dash='dot', line_color='gray',
              annotation_text='Fed 2% target', annotation_position='bottom right')

fig.update_layout(
    title=dict(text='US Inflation Over Time — When Did Your Dollar Lose Value?',
               font=dict(size=16)),
    xaxis_title='', yaxis_title='Inflation (Year-on-Year %)',
    hovermode='x unified',
    template='plotly_white',
    height=450
)
from IPython.display import display, HTML
display(HTML(fig.to_html()))
fig.write_html('chart2_inflation.html')
#print("Saved!")


In [ ]:
colors = df['SP500 Return (%)'].apply(lambda x: 'steelblue' if x >= 0 else 'crimson')

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df.index, y=df['SP500 Return (%)'],
    marker_color=colors,
    name='S&P 500 Monthly Return'
))

fig.update_layout(
    title=dict(text='S&P 500 Monthly Returns — Blue Means Up, Red Means Down',
               font=dict(size=16)),
    xaxis_title='', yaxis_title='Monthly Return (%)',
    hovermode='x unified',
    template='plotly_white',
    height=450
)
from IPython.display import display, HTML
display(HTML(fig.to_html()))
fig.write_html('chart3_sp500.html')
print("Saved!")

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['Oil Price ($/barrel)'],
    fill='tozeroy', fillcolor='rgba(128,128,0,0.15)',
    line=dict(color='olive', width=1.5),
    name='WTI Oil Price'
))

fig.update_layout(
    title=dict(text='Oil Price Per Barrel — The Economy\'s Heartbeat',
               font=dict(size=16)),
    xaxis_title='', yaxis_title='Price (USD per barrel)',
    hovermode='x unified',
    template='plotly_white',
    height=450
)

from IPython.display import display, HTML
display(HTML(fig.to_html()))
fig.write_html('chart4_oil.html')
print("Saved!")

In [ ]:
from google.colab import files
files.download('chart1_geopolitical_risk.html')
files.download('chart2_inflation.html')
files.download('chart3_sp500.html')
files.download('chart4_oil.html')
print("done")

In [ ]:
query1 = pd.read_sql("""
    SELECT
        strftime('%Y', "index") AS year,
        ROUND(AVG("Geopolitical Risk"), 1) AS avg_risk,
        ROUND(AVG("Inflation (%)"), 2) AS avg_inflation,
        ROUND(AVG("SP500 Return (%)"), 2) AS avg_sp500_return
    FROM macro_data
    GROUP BY year
    ORDER BY avg_risk DESC
    LIMIT 10
""", conn)

print("Top 10 years by geopolitical risk:")
print(query1.to_string(index=False))

In [ ]:
query2 = pd.read_sql("""
    SELECT
        CASE
            WHEN "Geopolitical Risk" > 200 THEN 'Very High Risk (>200)'
            WHEN "Geopolitical Risk" > 100 THEN 'High Risk (100-200)'
            ELSE 'Normal Risk (<100)'
        END AS risk_level,
        COUNT(*) AS months,
        ROUND(AVG("SP500 Return (%)"), 2) AS avg_stock_return,
        ROUND(AVG("Inflation (%)"), 2) AS avg_inflation,
        ROUND(AVG("Oil Return (%)"), 2) AS avg_oil_return
    FROM macro_data
    GROUP BY risk_level
    ORDER BY avg_stock_return
""", conn)

print("Market performance by risk level:")
print(query2.to_string(index=False))

In [ ]:
query3 = pd.read_sql("""
    SELECT
        strftime('%Y-%m', "index") AS month,
        ROUND("Geopolitical Risk", 1) AS risk,
        ROUND("Inflation (%)", 2) AS inflation,
        ROUND("SP500 Return (%)", 2) AS sp500_return,
        ROUND("Oil Price ($/barrel)", 2) AS oil_price
    FROM macro_data
    WHERE "index" BETWEEN '2022-01-01' AND '2022-12-31'
    ORDER BY "index"
""", conn)

print("Ukraine War year (2022) — month by month:")
print(query3.to_string(index=False))

In [ ]:
query4 = pd.read_sql("""
    SELECT
        strftime('%Y-%m', "index") AS month,
        ROUND("Inflation (%)", 2) AS inflation,
        ROUND("Geopolitical Risk", 1) AS risk,
        ROUND("SP500 Return (%)", 2) AS sp500_return
    FROM macro_data
    WHERE "Inflation (%)" > 5
    ORDER BY "Inflation (%)" DESC
    LIMIT 15
""", conn)

print("Months where inflation exceeded 5%:")
print(query4.to_string(index=False))

In [ ]:
fig = px.bar(
    query2,
    x='risk_level',
    y='avg_stock_return',
    color='avg_stock_return',
    color_continuous_scale=['crimson', 'lightgray', 'steelblue'],
    text='avg_stock_return',
    title='Average S&P 500 Return by Geopolitical Risk Level',
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(
    template='plotly_white',
    height=450,
    xaxis_title='Risk Level',
    yaxis_title='Avg Monthly Return (%)',
    coloraxis_showscale=False
)

from IPython.display import display, HTML
display(HTML(fig.to_html()))
fig.write_html('chart5_risk_vs_returns.html')
print("Saved!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import linregress
from numpy.linalg import lstsq
import sqlite3
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

conn = sqlite3.connect('geopolitical_risk.db')
df = pd.read_sql('SELECT * FROM macro_data', conn)
df = df.rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

print(f'Loaded {len(df)} rows | {df.index[0].date()} to {df.index[-1].date()}')
df.tail(3)


In [ ]:
GPR   = 'Geopolitical Risk'
CPI   = 'Inflation (%)'
SP500 = 'SP500 Return (%)'
OIL_R = 'Oil Return (%)'
OIL_P = 'Oil Price ($/barrel)'

MAX_LAG  = 24
ROLL_WIN = 36

CRISES = {
    'Gulf War':         '1990-08-01',
    '9/11':             '2001-09-01',
    'Iraq War':         '2003-03-01',
    'Financial Crisis': '2008-09-01',
    'Crimea':           '2014-03-01',
    'COVID-19':         '2020-03-01',
    'Ukraine War':      '2022-02-01',
}
CRISES_TS = {
    k: pd.Timestamp(v) for k, v in CRISES.items()
    if pd.Timestamp(v) >= df.index[0] and pd.Timestamp(v) <= df.index[-1]
}
print('Crisis dates confirmed:', list(CRISES_TS.keys()))

In [ ]:
for lag in range(1, MAX_LAG + 1):
    df[f'cpi_fwd_{lag}m']   = df[CPI].shift(-lag)
    df[f'sp500_fwd_{lag}m'] = df[SP500].shift(-lag)
    df[f'oil_fwd_{lag}m']   = df[OIL_R].shift(-lag)

df_c = df.dropna(subset=[GPR, CPI, SP500, OIL_R]).copy()
gpr_vals = df_c[GPR].values

print(f'Forward columns built | Clean rows: {len(df_c)}')

In [ ]:
def pearson_ci(x, y, alpha=0.05):
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    r, p = stats.pearsonr(x, y)
    z  = np.arctanh(r)
    se = 1 / np.sqrt(len(x) - 3)
    zc = stats.norm.ppf(1 - alpha / 2)
    return r, p, float(np.tanh(z - zc * se)), float(np.tanh(z + zc * se))

pairs = [
    ('GPR  vs  CPI Inflation',     df_c[CPI].values),
    ('GPR  vs  SP500 Return',      df_c[SP500].values),
    ('GPR  vs  Oil Return',        df_c[OIL_R].values),
    ('GPR  vs  Oil Price (level)', df_c[OIL_P].values),
]

print(f'{"Pair":<36} {"r":>7}  {"p-val":>8}  95% CI')
print('-' * 68)
for label, y_vals in pairs:
    r, p, lo, hi = pearson_ci(gpr_vals, y_vals)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'{label:<36} {r:+7.3f}  {p:8.4f}  [{lo:+.3f}, {hi:+.3f}] {sig}')

print('\n* p<0.05  ** p<0.01  *** p<0.001')

In [ ]:
lags = np.arange(0, MAX_LAG + 1)
res  = {'cpi': {'r':[],'p':[]}, 'sp500': {'r':[],'p':[]}, 'oil': {'r':[],'p':[]}}

for lag in lags:
    y_cpi   = df_c[CPI].values   if lag == 0 else df_c[f'cpi_fwd_{lag}m'].values
    y_sp500 = df_c[SP500].values if lag == 0 else df_c[f'sp500_fwd_{lag}m'].values
    y_oil   = df_c[OIL_R].values if lag == 0 else df_c[f'oil_fwd_{lag}m'].values

    for key, y in [('cpi', y_cpi), ('sp500', y_sp500), ('oil', y_oil)]:
        r, p, _, _ = pearson_ci(gpr_vals, y)
        res[key]['r'].append(r)
        res[key]['p'].append(p)

peak = {k: int(lags[np.argmax(np.abs(res[k]['r']))]) for k in res}

print('PEAK LAG SUMMARY')
for k, name in [('cpi','CPI'), ('sp500','SP500'), ('oil','Oil')]:
    pk = peak[k]
    print(f'  {name:<8} peak @ lag {pk:2d}m   r={res[k]["r"][pk]:+.3f}   p={res[k]["p"][pk]:.4f}')

In [ ]:
roll_cpi   = df_c[CPI].rolling(ROLL_WIN).corr(df_c[GPR])
roll_sp500 = df_c[SP500].rolling(ROLL_WIN).corr(df_c[GPR])
roll_oil   = df_c[OIL_R].rolling(ROLL_WIN).corr(df_c[GPR])

print(f'Rolling {ROLL_WIN}m stats:')
for name, s in [('GPR-CPI', roll_cpi), ('GPR-SP500', roll_sp500), ('GPR-Oil', roll_oil)]:
    print(f'  {name:<12} mean={s.mean():+.3f}  min={s.min():+.3f}  max={s.max():+.3f}')

In [ ]:
DARK_BG  = '#0f1117'
PANEL_BG = '#1a1d27'
TEXT_COL = '#e0e4f0'
GOLD     = '#f0c040'
TEAL     = '#3ec9c9'
OLIVE    = '#a8c060'
RED      = '#e05c5c'
MUTED    = '#7a8099'

def style_ax(ax):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=8)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    for sp in ax.spines.values():
        sp.set_edgecolor('#2e3145')
    ax.grid(True, color='#2a2e42', linewidth=0.5, linestyle='--')
    return ax

fig = plt.figure(figsize=(16, 18), facecolor=DARK_BG)
gs  = gridspec.GridSpec(4, 2, figure=fig,
                         hspace=0.48, wspace=0.35,
                         left=0.07, right=0.96, top=0.93, bottom=0.05)

# Panel A — Lag profile: GPR -> CPI
ax_a = style_ax(fig.add_subplot(gs[0, 0]))
sig_c = np.array(res['cpi']['p']) < 0.05
ax_a.bar(lags, res['cpi']['r'],
         color=[GOLD if s else MUTED for s in sig_c], width=0.7, alpha=0.85)
ax_a.axhline(0, color=TEXT_COL, lw=0.8, ls='--', alpha=0.5)
ax_a.axvline(peak['cpi'], color=GOLD, lw=1.5, ls=':', alpha=0.9)
ax_a.set_title('Lag Profile: GPR -> CPI Inflation', fontweight='bold')
ax_a.set_xlabel('Months forward (lag k)')
ax_a.set_ylabel('Pearson r')
ax_a.text(peak['cpi'] + 0.6, res['cpi']['r'][peak['cpi']] * 0.75,
          f"Peak @ {peak['cpi']}m", color=GOLD, fontsize=8)
ax_a.legend(handles=[Patch(color=GOLD, label='p<0.05'), Patch(color=MUTED, label='p>=0.05')],
            fontsize=7, facecolor=PANEL_BG, labelcolor=TEXT_COL, framealpha=0.6)

# Panel B — Lag profile: GPR -> SP500
ax_b = style_ax(fig.add_subplot(gs[0, 1]))
sig_s = np.array(res['sp500']['p']) < 0.05
ax_b.bar(lags, res['sp500']['r'],
         color=[TEAL if s else MUTED for s in sig_s], width=0.7, alpha=0.85)
ax_b.axhline(0, color=TEXT_COL, lw=0.8, ls='--', alpha=0.5)
ax_b.axvline(peak['sp500'], color=TEAL, lw=1.5, ls=':', alpha=0.9)
ax_b.set_title('Lag Profile: GPR -> SP500 Return', fontweight='bold')
ax_b.set_xlabel('Months forward (lag k)')
ax_b.set_ylabel('Pearson r')
r_sp_pk = res['sp500']['r'][peak['sp500']]
ax_b.text(peak['sp500'] + 0.6, r_sp_pk * 0.75 if r_sp_pk < 0 else r_sp_pk * 1.1,
          f"Peak @ {peak['sp500']}m", color=TEAL, fontsize=8)
ax_b.legend(handles=[Patch(color=TEAL, label='p<0.05'), Patch(color=MUTED, label='p>=0.05')],
            fontsize=7, facecolor=PANEL_BG, labelcolor=TEXT_COL, framealpha=0.6)

# Panel C — Scatter: GPR vs CPI at peak lag
ax_c = style_ax(fig.add_subplot(gs[1, 0]))
pk_c = peak['cpi']
y_c  = df_c[CPI].values if pk_c == 0 else df_c[f'cpi_fwd_{pk_c}m'].values
m    = ~(np.isnan(gpr_vals) | np.isnan(y_c))
sl, ic, rv, pv, _ = linregress(gpr_vals[m], y_c[m])
xl   = np.linspace(gpr_vals[m].min(), gpr_vals[m].max(), 200)
ax_c.scatter(gpr_vals[m], y_c[m], alpha=0.3, s=14, color=GOLD, edgecolors='none')
ax_c.plot(xl, ic + sl * xl, color=GOLD, lw=2.5)
ax_c.set_xlabel('Geopolitical Risk Index')
ax_c.set_ylabel(f'Inflation (%) +{pk_c}m forward')
ax_c.set_title(f'GPR vs CPI  (lag={pk_c}m)', fontweight='bold')
ax_c.text(0.05, 0.91, f'r={rv:.3f}  p={pv:.4f}', transform=ax_c.transAxes,
          fontsize=8, color=TEXT_COL,
          bbox=dict(boxstyle='round,pad=0.3', fc='#0f1117', alpha=0.7))

# Panel D — Scatter: GPR vs Oil at peak lag
ax_d = style_ax(fig.add_subplot(gs[1, 1]))
pk_o = peak['oil']
y_o  = df_c[OIL_R].values if pk_o == 0 else df_c[f'oil_fwd_{pk_o}m'].values
m2   = ~(np.isnan(gpr_vals) | np.isnan(y_o))
sl2, ic2, rv2, pv2, _ = linregress(gpr_vals[m2], y_o[m2])
xl2  = np.linspace(gpr_vals[m2].min(), gpr_vals[m2].max(), 200)
ax_d.scatter(gpr_vals[m2], y_o[m2], alpha=0.3, s=14, color=OLIVE, edgecolors='none')
ax_d.plot(xl2, ic2 + sl2 * xl2, color=OLIVE, lw=2.5)
ax_d.set_xlabel('Geopolitical Risk Index')
ax_d.set_ylabel(f'Oil Return (%) +{pk_o}m forward')
ax_d.set_title(f'GPR vs Oil Return  (lag={pk_o}m)', fontweight='bold')
ax_d.text(0.05, 0.91, f'r={rv2:.3f}  p={pv2:.4f}', transform=ax_d.transAxes,
          fontsize=8, color=TEXT_COL,
          bbox=dict(boxstyle='round,pad=0.3', fc='#0f1117', alpha=0.7))

# Panel E — Rolling correlation (full width)
ax_e = style_ax(fig.add_subplot(gs[2, :]))
ax_e.plot(roll_cpi.index,   roll_cpi.values,   color=GOLD,  lw=1.8, label=f'GPR-CPI ({ROLL_WIN}m)')
ax_e.plot(roll_sp500.index, roll_sp500.values, color=TEAL,  lw=1.8, label=f'GPR-SP500 ({ROLL_WIN}m)')
ax_e.plot(roll_oil.index,   roll_oil.values,   color=OLIVE, lw=1.8, ls='--', label=f'GPR-Oil ({ROLL_WIN}m)')
ax_e.axhline(0, color=TEXT_COL, lw=0.8, ls='--', alpha=0.4)
ax_e.fill_between(roll_cpi.index, 0, roll_cpi.values,
                   where=(roll_cpi.values > 0), alpha=0.10, color=GOLD)
ax_e.fill_between(roll_cpi.index, 0, roll_cpi.values,
                   where=(roll_cpi.values < 0), alpha=0.10, color=RED)
y_top = ax_e.get_ylim()[1]
for label, ts in CRISES_TS.items():
    ax_e.axvline(ts, color=RED, lw=0.9, ls=':', alpha=0.6)
    ax_e.text(ts, y_top * 0.88, label,
              rotation=90, fontsize=6.5, color=RED, va='top', ha='right')
ax_e.set_title(f'Rolling {ROLL_WIN}-Month Correlation: GPR vs CPI, SP500 & Oil', fontweight='bold')
ax_e.set_ylabel('Pearson r (rolling)')
ax_e.legend(fontsize=7.5, facecolor=PANEL_BG, labelcolor=TEXT_COL, framealpha=0.6, loc='lower right')

# Panel F — Heatmap (full width)
ax_f = style_ax(fig.add_subplot(gs[3, :]))
corr_matrix = np.array([res['oil']['r'], res['sp500']['r'], res['cpi']['r']])
im = ax_f.imshow(corr_matrix, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5,
                  extent=[-0.5, MAX_LAG + 0.5, -0.5, 2.5])
ax_f.set_yticks([0, 1, 2])
ax_f.set_yticklabels(['CPI', 'SP500', 'Oil Ret'], color=TEXT_COL, fontsize=9)
ax_f.set_xticks(range(0, MAX_LAG + 1, 3))
ax_f.set_xlabel('Lag k (months forward)', color=TEXT_COL)
ax_f.set_title('Correlation Heatmap: GPR vs All Outcomes across Lags  (* = p<0.05)', fontweight='bold')
for i, key in enumerate(['cpi', 'sp500', 'oil']):
    for j, p_v in enumerate(res[key]['p']):
        if p_v < 0.05:
            ax_f.text(j, i, '*', ha='center', va='center', fontsize=9, color='white')
cbar = plt.colorbar(im, ax=ax_f, orientation='vertical', fraction=0.015, pad=0.02)
cbar.ax.tick_params(colors=TEXT_COL, labelsize=7)
cbar.set_label('Pearson r', color=TEXT_COL, fontsize=8)

fig.text(0.5, 0.965, 'Geopolitical Risk — Correlation & Lag Analysis',
         ha='center', va='top', fontsize=14, fontweight='bold', color=TEXT_COL, fontfamily='serif')
fig.text(0.5, 0.950, '* = p<0.05  |  Gold = CPI  |  Teal = SP500  |  Olive = Oil',
         ha='center', va='top', fontsize=9, color=MUTED)

plt.savefig('/content/section4_correlation_lag.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# chart6 — Lag profiles
fig6 = make_subplots(rows=1, cols=3,
    subplot_titles=('GPR -> CPI', 'GPR -> SP500', 'GPR -> Oil'))

for col_idx, (key, color) in enumerate(
        [('cpi','gold'), ('sp500','teal'), ('oil','olivedrab')], 1):
    bar_colors = [color if p < 0.05 else 'lightgray' for p in res[key]['p']]
    fig6.add_trace(go.Bar(
        x=list(lags), y=res[key]['r'],
        marker_color=bar_colors,
        hovertemplate='Lag %{x}m<br>r=%{y:.3f}<extra></extra>',
        showlegend=False
    ), row=1, col=col_idx)
    fig6.add_vline(x=peak[key], line_dash='dot', line_color=color,
                   opacity=0.8, row=1, col=col_idx)

fig6.update_layout(
    title='How Far Ahead Does GPR Predict Each Outcome? (colored = p<0.05)',
    template='plotly_white', height=400, hovermode='x unified')
fig6.update_xaxes(title_text='Lag (months forward)')
fig6.update_yaxes(title_text='Pearson r', col=1)

display(HTML(fig6.to_html()))
fig6.write_html('chart6_lag_correlations.html')
print('Saved: chart6_lag_correlations.html')

# chart7 — Rolling correlation
fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=roll_cpi.index,   y=roll_cpi.values,
    line=dict(color='gold', width=2),     name='GPR vs CPI'))
fig7.add_trace(go.Scatter(x=roll_sp500.index, y=roll_sp500.values,
    line=dict(color='teal', width=2),     name='GPR vs SP500'))
fig7.add_trace(go.Scatter(x=roll_oil.index,   y=roll_oil.values,
    line=dict(color='olivedrab', width=2, dash='dash'), name='GPR vs Oil'))
fig7.add_hline(y=0, line_dash='dot', line_color='gray', opacity=0.5)

for label, ts in CRISES_TS.items():
    fig7.add_vline(x=ts.isoformat(), line_dash='dash',
                   line_color='black', line_width=0.8, opacity=0.5)
    fig7.add_annotation(x=ts, y=0.95, yref='paper', text=label,
                        showarrow=False, textangle=-90,
                        font=dict(size=9), xanchor='right')

fig7.update_layout(
    title='Rolling 36-Month Correlation: GPR vs Markets',
    yaxis_title='Pearson r (rolling 36m)',
    template='plotly_white', height=460, hovermode='x unified')

display(HTML(fig7.to_html()))
fig7.write_html('chart7_rolling_correlation.html')
print('Saved: chart7_rolling_correlation.html')

In [ ]:
def stars(p):
    return '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '-'

rows = []
for lag in [0, 3, 6, 9, 12, 18, 24]:
    y_c = df_c[CPI].values   if lag == 0 else df_c[f'cpi_fwd_{lag}m'].values
    y_s = df_c[SP500].values if lag == 0 else df_c[f'sp500_fwd_{lag}m'].values
    y_o = df_c[OIL_R].values if lag == 0 else df_c[f'oil_fwd_{lag}m'].values
    r_c, p_c, _, _ = pearson_ci(gpr_vals, y_c)
    r_s, p_s, _, _ = pearson_ci(gpr_vals, y_s)
    r_o, p_o, _, _ = pearson_ci(gpr_vals, y_o)
    rows.append({
        'Lag (m)': lag,
        'r(GPR,CPI)': f'{r_c:+.3f}',    'CPI': stars(p_c),
        'r(GPR,SP500)': f'{r_s:+.3f}',  'SP500': stars(p_s),
        'r(GPR,Oil)': f'{r_o:+.3f}',    'Oil': stars(p_o),
    })

print(pd.DataFrame(rows).to_string(index=False))
print('\n* p<0.05  ** p<0.01  *** p<0.001')

In [ ]:
r_a, p_a, _, _ = pearson_ci(df_c[GPR].values, df_c[OIL_R].values)
r_b, p_b, _, _ = pearson_ci(df_c[OIL_R].values, df_c['cpi_fwd_6m'].values)
r_d, p_d, _, _ = pearson_ci(gpr_vals, df_c['cpi_fwd_6m'].values)

oil_v  = df_c[OIL_R].values
cpi_6m = df_c['cpi_fwd_6m'].values
mask   = ~(np.isnan(gpr_vals) | np.isnan(oil_v) | np.isnan(cpi_6m))
X      = np.column_stack([np.ones(mask.sum()), oil_v[mask]])
coef, _, _, _ = lstsq(X, cpi_6m[mask], rcond=None)
resid  = cpi_6m[mask] - (coef[0] + coef[1] * oil_v[mask])
r_part, p_part = stats.pearsonr(gpr_vals[mask], resid)

print('MEDIATION CHECK: GPR -> Oil -> CPI (lag 6m)')
print(f'  Path A   GPR -> Oil (lag 0)       r={r_a:+.3f}  p={p_a:.4f}')
print(f'  Path B   Oil -> CPI (lag 6m)      r={r_b:+.3f}  p={p_b:.4f}')
print(f'  Direct   GPR -> CPI (lag 6m)      r={r_d:+.3f}  p={p_d:.4f}')
print(f'  Partial  GPR -> CPI | Oil removed r={r_part:+.3f}  p={p_part:.4f}')

if abs(r_part) < abs(r_d) and p_a < 0.05 and p_b < 0.05:
    pct = (abs(r_d) - abs(r_part)) / abs(r_d) * 100
    print(f'\nOil mediates ~{pct:.0f}% of the GPR->CPI relationship.')

In [ ]:
from google.colab import files
files.download('/content/section4_correlation_lag.png')
files.download('chart6_lag_correlations.html')
files.download('chart7_rolling_correlation.html')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import sqlite3
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

conn = sqlite3.connect('geopolitical_risk.db')
df = pd.read_sql('SELECT * FROM macro_data', conn)
df = df.rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

print(f'Loaded {len(df)} rows | {df.index[0].date()} to {df.index[-1].date()}')

In [ ]:
GPR   = 'Geopolitical Risk'
CPI   = 'Inflation (%)'
SP500 = 'SP500 Return (%)'
OIL_R = 'Oil Return (%)'
OIL_P = 'Oil Price ($/barrel)'

# Each event: label -> (date, one-line description for chart)
EVENTS = {
    'Gulf War':         ('1990-08-01', 'Iraq invades Kuwait'),
    '9/11':             ('2001-09-01', 'Terror attacks on US'),
    'Iraq War':         ('2003-03-01', 'US-led invasion of Iraq'),
    'Financial Crisis': ('2008-09-01', 'Lehman Brothers collapse'),
    'Crimea':           ('2014-03-01', 'Russia annexes Crimea'),
    'COVID-19':         ('2020-03-01', 'Global pandemic declared'),
    'Ukraine War':      ('2022-02-01', 'Russia invades Ukraine'),
}

PRE_WINDOW  = 36   # months before event used to estimate baseline
EVENT_PRE   = 3    # months before t=0 to show on chart
EVENT_POST  = 18   # months after t=0 to show on chart

# Filter to events inside our data range
EVENTS = {
    k: v for k, v in EVENTS.items()
    if pd.Timestamp(v[0]) >= df.index[0]
    and pd.Timestamp(v[0]) <= df.index[-1]
}
print(f'{len(EVENTS)} events in data range:')
for k, (d, desc) in EVENTS.items():
    print(f'  {k:<20} {d}  —  {desc}')

In [ ]:
def run_event_study(series, event_date_str, pre_window=36,
                    event_pre=3, event_post=18):
    """
    Compute an event study for a single series around a single event date.

    Parameters
    ----------
    series         : pd.Series with a monthly DatetimeIndex
    event_date_str : 'YYYY-MM-DD' string for t=0
    pre_window     : months before the event used to estimate the baseline
                     (estimation window = [t - pre_window - event_pre, t - event_pre])
    event_pre      : months before t=0 to include in the event window
    event_post     : months after  t=0 to include in the event window

    Returns
    -------
    dict with keys:
        rel_idx    : array of integers relative to event (-event_pre ... +event_post)
        actual     : actual series values in the event window
        baseline   : scalar mean of the estimation window
        abnormal   : actual - baseline  (Abnormal Value)
        cum_abn    : cumulative sum of abnormal values (CAR for returns)
        t_stats    : t-statistic for each abnormal value vs estimation window std
        sig        : boolean array — is |t_stat| > 1.96?
        est_std    : std of the estimation window (used for t-stats)
    """
    event_dt = pd.Timestamp(event_date_str)

    # Find the closest month in the index to the event date
    t0 = series.index.get_indexer([event_dt], method='nearest')[0]

    # Estimation window indices
    est_start = max(0, t0 - pre_window - event_pre)
    est_end   = t0 - event_pre

    # Event window indices
    evt_start = max(0, t0 - event_pre)
    evt_end   = min(len(series) - 1, t0 + event_post)

    if est_end <= est_start + 5:   # need at least 6 obs to estimate baseline
        return None

    est_vals = series.iloc[est_start:est_end].dropna()
    evt_vals = series.iloc[evt_start:evt_end + 1]

    baseline = est_vals.mean()
    est_std  = est_vals.std()

    rel_idx  = np.arange(-event_pre, len(evt_vals) - event_pre)
    abnormal = evt_vals.values - baseline
    cum_abn  = np.cumsum(abnormal)
    t_stats  = abnormal / est_std if est_std > 0 else np.zeros_like(abnormal)
    sig      = np.abs(t_stats) > 1.96   # ~95% confidence

    return {
        'rel_idx'  : rel_idx,
        'actual'   : evt_vals.values,
        'baseline' : baseline,
        'abnormal' : abnormal,
        'cum_abn'  : cum_abn,
        't_stats'  : t_stats,
        'sig'      : sig,
        'est_std'  : est_std,
        'event_dt' : event_dt,
    }

print('Event study function defined.')
print(f'Estimation window : {PRE_WINDOW} months before event')
print(f'Event window      : t={-EVENT_PRE} to t=+{EVENT_POST} months')

In [ ]:
results = {}   # results[event_name][variable] = event study dict

for event_name, (event_date, _) in EVENTS.items():
    results[event_name] = {}
    for var_name, series in [(SP500, df[SP500]), (CPI, df[CPI]),
                              (OIL_R, df[OIL_R]), (GPR, df[GPR])]:
        out = run_event_study(series, event_date,
                              PRE_WINDOW, EVENT_PRE, EVENT_POST)
        results[event_name][var_name] = out

# Quick sanity check — print CAR at +3m and +12m for SP500
print(f'{"Event":<22} {"CAR +3m":>9} {"CAR +6m":>9} {"CAR +12m":>10}')
print('-' * 54)
for event_name in results:
    r = results[event_name][SP500]
    if r is None:
        continue
    def car_at(r, lag):
        idx = np.where(r['rel_idx'] == lag)[0]
        return f'{r["cum_abn"][idx[0]] * 100:+.1f}%' if len(idx) else 'n/a'
    print(f'{event_name:<22} {car_at(r,3):>9} {car_at(r,6):>9} {car_at(r,12):>10}')

In [ ]:
DARK_BG  = '#0f1117'
PANEL_BG = '#1a1d27'
TEXT_COL = '#e0e4f0'
GOLD     = '#f0c040'
TEAL     = '#3ec9c9'
OLIVE    = '#a8c060'
RED      = '#e05c5c'
GREEN    = '#5ce08a'
MUTED    = '#7a8099'

def style_ax(ax):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=8)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)
    for sp in ax.spines.values():
        sp.set_edgecolor('#2e3145')
    ax.grid(True, color='#2a2e42', linewidth=0.5, linestyle='--')
    return ax

n = len(EVENTS)
ncols = 2
nrows = (n + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 5), facecolor=DARK_BG)
axes = axes.flatten()

for i, (event_name, (event_date, description)) in enumerate(EVENTS.items()):
    ax = style_ax(axes[i])
    ax2 = ax.twinx()
    ax2.set_facecolor('none')
    ax2.tick_params(colors=GOLD, labelsize=7)
    ax2.yaxis.label.set_color(GOLD)
    for sp in ax2.spines.values():
        sp.set_edgecolor('#2e3145')

    r_sp  = results[event_name][SP500]
    r_cpi = results[event_name][CPI]

    if r_sp is None:
        ax.set_title(f'{event_name} — insufficient data', color=TEXT_COL)
        continue

    rel = r_sp['rel_idx']

    # Bars — monthly abnormal SP500 return
    bar_colors = [GREEN if v >= 0 else RED for v in r_sp['abnormal']]
    ax.bar(rel, r_sp['abnormal'] * 100, color=bar_colors,
           alpha=0.65, width=0.7, label='Monthly abnormal return (%)')

    # Teal line — Cumulative Abnormal Return
    ax.plot(rel, r_sp['cum_abn'] * 100, color=TEAL,
            lw=2.2, zorder=3, label='CAR (%)')

    # Significance dots on CAR line
    sig_idx = np.where(r_sp['sig'])[0]
    if len(sig_idx):
        ax.scatter(rel[sig_idx], r_sp['cum_abn'][sig_idx] * 100,
                   color=TEAL, s=40, zorder=4)

    # Gold dashed — CPI abnormal deviation (right axis)
    if r_cpi is not None:
        rel_c = r_cpi['rel_idx']
        ax2.plot(rel_c, r_cpi['abnormal'], color=GOLD,
                 lw=1.8, ls='--', label='CPI deviation from baseline')
        ax2.set_ylabel('CPI deviation (pp)', fontsize=7)

    # Event line
    ax.axvline(0, color=RED, lw=1.5, ls='--', alpha=0.9)
    ax.axhline(0, color=TEXT_COL, lw=0.6, ls='--', alpha=0.3)

    # Annotate CAR at +12m if available
    idx_12 = np.where(rel == min(12, rel[-1]))[0]
    if len(idx_12):
        car_12 = r_sp['cum_abn'][idx_12[0]] * 100
        ax.annotate(f'CAR+12m\n{car_12:+.1f}%',
                    xy=(rel[idx_12[0]], car_12),
                    xytext=(rel[idx_12[0]] - 3, car_12 * 1.3 if car_12 != 0 else 2),
                    fontsize=7, color=TEAL,
                    arrowprops=dict(arrowstyle='->', color=TEAL, lw=1))

    ax.set_title(f'{event_name}  —  {description}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Months relative to event (t=0)')
    ax.set_ylabel('SP500 return % (bars=monthly, line=CAR)', color=TEAL, fontsize=7)

    # Merged legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=6.5, facecolor=PANEL_BG,
              labelcolor=TEXT_COL, framealpha=0.6, loc='lower left')

# Hide any unused panels
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Event Study — Market & Inflation Response Around Geopolitical Crises',
             fontsize=13, fontweight='bold', color=TEXT_COL,
             fontfamily='serif', y=1.01)
fig.text(0.5, 0.995,
         'Teal = Cumulative Abnormal Return  |  Bars = Monthly AR  '
         '|  Gold dashed = CPI deviation  |  Dot = significant at 95%',
         ha='center', fontsize=8, color=MUTED)

plt.tight_layout(h_pad=3.5, w_pad=2.5)
plt.savefig('/content/section5_event_study.png',
            dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('Saved: /content/section5_event_study.png')

In [ ]:
fig8 = go.Figure()

event_names = list(EVENTS.keys())
first = True

for event_name, (event_date, description) in EVENTS.items():
    r_sp  = results[event_name][SP500]
    r_cpi = results[event_name][CPI]
    r_oil = results[event_name][OIL_R]
    if r_sp is None:
        continue

    visible = True if first else False
    rel = r_sp['rel_idx']
    bar_colors = ['#5ce08a' if v >= 0 else '#e05c5c' for v in r_sp['abnormal']]

    # Monthly abnormal return bars
    fig8.add_trace(go.Bar(
        x=rel, y=r_sp['abnormal'] * 100,
        marker_color=bar_colors,
        name='Monthly Abnormal Return (%)',
        visible=visible,
        hovertemplate='t=%{x}m<br>Abnormal return: %{y:.2f}%<extra></extra>'
    ))

    # CAR line
    fig8.add_trace(go.Scatter(
        x=rel, y=r_sp['cum_abn'] * 100,
        line=dict(color='teal', width=2.5),
        name='Cumulative Abnormal Return (%)',
        visible=visible,
        hovertemplate='t=%{x}m<br>CAR: %{y:.2f}%<extra></extra>'
    ))

    # CPI deviation
    if r_cpi is not None:
        fig8.add_trace(go.Scatter(
            x=r_cpi['rel_idx'], y=r_cpi['abnormal'],
            line=dict(color='gold', width=2, dash='dash'),
            name='CPI deviation (pp)',
            visible=visible,
            yaxis='y2',
            hovertemplate='t=%{x}m<br>CPI dev: %{y:.2f}pp<extra></extra>'
        ))

    # Oil abnormal return
    if r_oil is not None:
        fig8.add_trace(go.Scatter(
            x=r_oil['rel_idx'], y=r_oil['abnormal'] * 100,
            line=dict(color='olivedrab', width=1.8, dash='dot'),
            name='Oil abnormal return (%)',
            visible=visible,
            hovertemplate='t=%{x}m<br>Oil AR: %{y:.2f}%<extra></extra>'
        ))

    first = False

# Build dropdown buttons
traces_per_event = 4   # bars + CAR + CPI + Oil
buttons = []
for i, (event_name, (event_date, description)) in enumerate(EVENTS.items()):
    vis = [False] * (len(EVENTS) * traces_per_event)
    for j in range(traces_per_event):
        idx = i * traces_per_event + j
        if idx < len(vis):
            vis[idx] = True
    buttons.append(dict(
        label=event_name,
        method='update',
        args=[
            {'visible': vis},
            {'title': f'Event Study: {event_name} ({event_date})<br>'
                      f'<sup>{description}</sup>'}
        ]
    ))

fig8.update_layout(
    title=f'Event Study: {event_names[0]} — use dropdown to switch events',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0.01, xanchor='left',
        y=1.15, yanchor='top',
        bgcolor='white',
        bordercolor='gray',
        font=dict(size=11)
    )],
    xaxis=dict(title='Months relative to event (t=0)', zeroline=True,
               zerolinecolor='crimson', zerolinewidth=1.5),
    yaxis=dict(title='SP500 return / CAR (%)'),
    yaxis2=dict(title='CPI deviation (pp)', overlaying='y',
                side='right', showgrid=False),
    template='plotly_white',
    height=500,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

display(HTML(fig8.to_html()))
fig8.write_html('chart8_event_study.html')
print('Saved: chart8_event_study.html')

In [ ]:
horizons = [1, 3, 6, 9, 12, 18]
event_cars = {h: [] for h in horizons}

for event_name in results:
    r = results[event_name][SP500]
    if r is None:
        continue
    for h in horizons:
        idx = np.where(r['rel_idx'] == h)[0]
        if len(idx):
            event_cars[h].append(r['cum_abn'][idx[0]] * 100)

avg_cars = {h: np.mean(v) for h, v in event_cars.items() if v}
std_cars = {h: np.std(v)  for h, v in event_cars.items() if v}

print('AVERAGE CAR ACROSS ALL EVENTS')
print('-' * 40)
for h in horizons:
    if h in avg_cars:
        print(f'  +{h:2d}m   avg CAR = {avg_cars[h]:+.2f}%   '
              f'std = {std_cars[h]:.2f}%   n = {len(event_cars[h])}')

# --- Chart 9: avg CAR bar chart ---
fig9 = go.Figure()

avg_vals = [avg_cars.get(h, 0) for h in horizons]
std_vals = [std_cars.get(h, 0) for h in horizons]
bar_cols  = ['#5ce08a' if v >= 0 else '#e05c5c' for v in avg_vals]

fig9.add_trace(go.Bar(
    x=[f'+{h}m' for h in horizons],
    y=avg_vals,
    error_y=dict(type='data', array=std_vals, visible=True,
                 color='rgba(255,255,255,0.5)'),
    marker_color=bar_cols,
    text=[f'{v:+.1f}%' for v in avg_vals],
    textposition='outside',
    hovertemplate='Horizon %{x}<br>Avg CAR: %{y:.2f}%<extra></extra>'
))

fig9.update_layout(
    title=dict(
        text='Average Cumulative Abnormal Return After a Geopolitical Shock<br>'
             '<sup>Averaged across all events in dataset  |  Error bars = 1 std dev across events</sup>',
        font=dict(size=15)
    ),
    xaxis_title='Horizon (months after event)',
    yaxis_title='Average CAR (%)',
    template='plotly_white',
    height=420,
    showlegend=False
)
fig9.add_hline(y=0, line_dash='dot', line_color='gray', opacity=0.5)

display(HTML(fig9.to_html()))
fig9.write_html('chart9_avg_car.html')
print('Saved: chart9_avg_car.html')

In [ ]:
rows = []
for event_name, (event_date, description) in EVENTS.items():
    event_dt = pd.Timestamp(event_date)
    idx = df.index.get_indexer([event_dt], method='nearest')[0]

    window = df.iloc[max(0, idx-3) : idx+4]
    gpr_pre  = df[GPR].iloc[max(0,idx-6):idx].mean()
    gpr_peak = df[GPR].iloc[idx]
    sp_m0    = df[SP500].iloc[idx]
    cpi_m0   = df[CPI].iloc[idx]
    oil_m0   = df[OIL_R].iloc[idx]

    rows.append({
        'Event':           event_name,
        'Date':            event_date[:7],
        'GPR (pre-avg)':   f'{gpr_pre:.0f}',
        'GPR (t=0)':       f'{gpr_peak:.0f}',
        'GPR spike':       f'+{gpr_peak - gpr_pre:.0f}',
        'SP500 t=0 (%)':   f'{sp_m0:+.1f}',
        'Oil t=0 (%)':     f'{oil_m0:+.1f}',
        'CPI t=0 (%)':     f'{cpi_m0:.1f}',
    })

tbl = pd.DataFrame(rows)
print('CRISIS SNAPSHOT TABLE')
print('=' * 80)
print(tbl.to_string(index=False))
print()
print('GPR spike = GPR at t=0 minus average GPR in 6 months before event')
print('This table is ready to paste into your README or report.')

In [ ]:
from google.colab import files
files.download('/content/section5_event_study.png')
files.download('chart8_event_study.html')
files.download('chart9_avg_car.html')
print('Done. Files downloaded.')

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Load
conn = sqlite3.connect('geopolitical_risk.db')
df = pd.read_sql('SELECT * FROM macro_data', conn)
df = df.rename(columns={'index': 'date'})
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Derived columns
df['GPR_12m_Avg']          = df['Geopolitical Risk'].rolling(12).mean()
df['SP500_Index']          = (1 + df['SP500 Return (%)'] / 100).cumprod() * 100
df['Roll_Corr_GPR_CPI']    = df['Inflation (%)'].rolling(36).corr(df['Geopolitical Risk'])
df['Roll_Corr_GPR_SP500']  = df['SP500 Return (%)'].rolling(36).corr(df['Geopolitical Risk'])
df['Roll_Corr_GPR_Oil']    = df['Oil Return (%)'].rolling(36).corr(df['Geopolitical Risk'])

df['GPR_Tier'] = pd.cut(
    df['Geopolitical Risk'],
    bins=[-np.inf, 100, 200, np.inf],
    labels=['Normal', 'High', 'Very High']
)

CRISES = {
    'Gulf War':         '1990-08-01',
    '9/11':             '2001-09-01',
    'Iraq War':         '2003-03-01',
    'Financial Crisis': '2008-09-01',
    'Crimea':           '2014-03-01',
    'COVID-19':         '2020-03-01',
    'Ukraine War':      '2022-02-01',
}
df['Crisis_Label'] = ''
for label, date in CRISES.items():
    ts = pd.Timestamp(date)
    nearest = df.index[df.index.get_indexer([ts], method='nearest')[0]]
    df.at[nearest, 'Crisis_Label'] = label

# Clean column names — no special characters
df = df.reset_index()
df = df.rename(columns={
    'date'                 : 'Date',
    'Geopolitical Risk'    : 'GPR',
    'Inflation (%)'        : 'Inflation_Pct',
    'SP500 Return (%)'     : 'SP500_Return',
    'Oil Return (%)'       : 'Oil_Return',
    'Oil Price ($/barrel)' : 'Oil_Price',
})

# Export
df.to_csv('tableau_data.csv', index=False,
          encoding='utf-8-sig', date_format='%Y-%m-%d')

print('Columns:', list(df.columns))
print('Rows:', len(df))
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
df.head(3)

In [ ]:
from google.colab import files
files.download('tableau_data.csv')
#files.download('tableau_event_study.csv')
print('Downloaded')